# EDA — IBM AML HI-Small Dataset

**Week 1, Day 1.** Bootstrap for subsequent EDA work. Mounts Drive, downloads dataset, prepares paths.

**Dataset source:** [IBM Transactions for Anti-Money Laundering (AML)](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml)

In [11]:
# ============================================================
# Bootstrap — run once per Colab session
# ============================================================
# Mounts Drive, installs packages, sets paths, downloads data.
# Idempotent: safe to run every session.

from google.colab import drive
drive.mount('/content/drive')

import os
import pathlib
import zipfile

DATA_ROOT = pathlib.Path('/content/drive/MyDrive/fincrime-sentinel-data')
RAW = DATA_ROOT / 'raw'
PROCESSED = DATA_ROOT / 'processed'
TYPOLOGY = DATA_ROOT / 'typology_guidance'
SANCTIONS = DATA_ROOT / 'sanctions'

for path in [RAW, PROCESSED, TYPOLOGY, SANCTIONS]:
    path.mkdir(parents=True, exist_ok=True)

# Kaggle auth
os.environ['KAGGLE_CONFIG_DIR'] = str(DATA_ROOT)
kaggle_token = DATA_ROOT / 'kaggle.json'
assert kaggle_token.exists(), (
    f"Upload kaggle.json to {DATA_ROOT} before continuing. "
    "Get it from kaggle.com → Settings → Create New API Token."
)
os.chmod(kaggle_token, 0o600)

# Install non-default packages
!pip install -q kaggle pyarrow duckdb networkx

# Download dataset if not already in Drive
trans_file = RAW / 'HI-Small_Trans.csv'
patterns_file = RAW / 'HI-Small_Patterns.txt'

if not trans_file.exists():
    print("Downloading IBM AML dataset zip (~7.6GB, 5-10 min)...")
    !kaggle datasets download \
        -d ealtman2019/ibm-transactions-for-anti-money-laundering-aml \
        -p {RAW}

    zip_path = RAW / 'ibm-transactions-for-anti-money-laundering-aml.zip'

    print("Extracting HI-Small files only...")

    # Selectively extract only the two files
    with zipfile.ZipFile(zip_path, 'r') as z:
        all_files = z.namelist()
        print(f"Files in zip: {all_files}")

        for target in ['HI-Small_Trans.csv', 'HI-Small_Patterns.txt']:
            match = [f for f in all_files if target in f]
            if match:
                print(f"Extracting {match[0]}...")
                z.extract(match[0], path=RAW)
            else:
                print(f"WARNING: {target} not found in zip")

    print("Deleting zip to free space...")
    zip_path.unlink()
    print(f"Deleted {zip_path}")

else:
    print(f"Dataset already in Drive at {RAW}")

# Verify both files exist
assert trans_file.exists(), "HI-Small_Trans.csv missing after download"
patterns_file = RAW / 'HI-Small_Patterns.txt'
assert patterns_file.exists(), "HI-Small_Patterns.txt missing after download"

trans_size_mb = trans_file.stat().st_size / (1024 * 1024)
print(f"\n✓ Bootstrap complete")
print(f"✓ Transactions file: {trans_size_mb:.1f} MB")
print(f"✓ Patterns file: {patterns_file.stat().st_size} bytes")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset URL: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml
License(s): Community Data License Agreement - Sharing - Version 1.0
100% 7.61G/7.61G [05:10<00:00, 26.3MB/s]

Extracting HI-Small files only...
Files in zip: ['HI-Large_Patterns.txt', 'HI-Large_Trans.csv', 'HI-Large_accounts.csv', 'HI-Medium_Patterns.txt', 'HI-Medium_Trans.csv', 'HI-Medium_accounts.csv', 'HI-Small_Patterns.txt', 'HI-Small_Trans.csv', 'HI-Small_accounts.csv', 'LI-Large_Patterns.txt', 'LI-Large_Trans.csv', 'LI-Large_accounts.csv', 'LI-Medium_Patterns.txt', 'LI-Medium_Trans.csv', 'LI-Medium_accounts.csv', 'LI-Small_Patterns.txt', 'LI-Small_Trans.csv', 'LI-Small_accounts.csv']
Extracting HI-Small_Trans.csv...
Extracting HI-Small_Patterns.txt...
Deleting zip to free space...
Deleted /content/drive/MyDrive/fincrime-sentinel-data/raw/ibm-transact

In [12]:
import pathlib
RAW = pathlib.Path('/content/drive/MyDrive/fincrime-sentinel-data/raw')
for f in sorted(RAW.iterdir()):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"{f.name}: {size_mb:.1f} MB")

HI-Small_Patterns.txt: 0.3 MB
HI-Small_Trans.csv: 453.6 MB
